# 16 · Dates & Times

Timestamps are everywhere in data — and a common source of bugs. This notebook
covers `date`, `datetime`, `timedelta`, parsing/formatting strings, epoch time,
and why timezones matter.

## `date`, `datetime`, `timedelta`

`date` is a calendar day; `datetime` adds time of day; `timedelta` is a
duration you add or subtract. Differences between datetimes give timedeltas.

In [ ]:
from datetime import date, datetime, timedelta

today = date(2024, 6, 15)
print('today:', today, '| weekday:', today.strftime('%A'))
print('in 30 days:', today + timedelta(days=30))

start = datetime(2024, 6, 1, 9, 0, 0)
end = datetime(2024, 6, 1, 17, 30, 0)
dur = end - start
print('duration:', dur, '=', dur.total_seconds() / 3600, 'hours')

## Parsing strings → datetimes (`strptime`)

Ingested timestamps are strings. `strptime` parses them using **format codes**
(`%Y` year, `%m` month, `%d` day, `%H:%M:%S` time). Our data uses ISO-8601, and
`fromisoformat` is the fast path for that.

In [ ]:
from datetime import datetime

s = '2024-06-01 14:30:00'
dt = datetime.strptime(s, '%Y-%m-%d %H:%M:%S')
print(dt, '| year:', dt.year, '| hour:', dt.hour)

iso = '2024-06-01T14:30:00'
print('fromisoformat:', datetime.fromisoformat(iso))

## Formatting datetimes → strings (`strftime`)

The reverse: turn a datetime into a string for filenames, partitions, or
reports. Partition paths like `year=2024/month=06/day=01` come straight from
`strftime`.

In [ ]:
from datetime import datetime

dt = datetime(2024, 6, 1, 14, 30)
print(dt.strftime('%Y-%m-%d'))
print(dt.strftime('%Y%m%d_%H%M%S'))            # good for filenames
print(dt.strftime('year=%Y/month=%m/day=%d'))  # Hive-style partition

## Real example: bucket orders by month

Parse the `order_ts` strings from the data and count orders per month — the kind
of time-bucketing you do constantly.

In [ ]:
import csv
from collections import Counter
from datetime import datetime
from pathlib import Path

def find_data():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('run data/build_data.py')
RAW = find_data() / 'raw'

by_month = Counter()
with open(RAW / 'orders.csv', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):
        dt = datetime.fromisoformat(row['order_ts'])
        by_month[dt.strftime('%Y-%m')] += 1
for month, n in sorted(by_month.items())[:6]:
    print(month, n)

## Epoch time and timezones

**Epoch** (Unix) time is seconds since 1970-01-01 UTC — how systems exchange
instants unambiguously. A **naive** datetime has no timezone; an **aware** one
does. Rule of thumb: store and compute in **UTC**, convert to local only for
display.

In [ ]:
from datetime import datetime, timezone, timedelta

aware = datetime(2024, 6, 1, 14, 30, tzinfo=timezone.utc)
print('utc:', aware)
print('epoch seconds:', aware.timestamp())
print('back from epoch:', datetime.fromtimestamp(aware.timestamp(), tz=timezone.utc))

ist = timezone(timedelta(hours=5, minutes=30))
print('same instant in IST:', aware.astimezone(ist))

### Recap

`date`/`datetime`/`timedelta` model days, instants and durations; `strptime`
parses, `strftime` formats (and builds partition paths); `fromisoformat` is the
fast ISO path; store/compute in UTC and treat epoch as the interchange format.
Next: talking to HTTP APIs.